In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv('.env.openai')) # read local .env file

In [2]:
import miniflux
# Authentication using an API token
miniflux_client = miniflux.Client(os.getenv('MINIFLUX_API_BASE'), api_key=os.getenv('MINIFLUX_API_KEY'))

# litellm.api_key = os.getenv('LITELLM_API_KEY')
# litellm.api_base = os.getenv('LITELLM_ROUTER')
from openai import OpenAI

client = OpenAI(api_key=os.environ.get("LITELLM_API_KEY"), base_url=os.getenv('LITELLM_ROUTER'))
def system(text):
    return {'content': text, "role": "system"}
def user(text):
    return {'content': text, "role": "user"}
def chat(model, messages, **kvargs):
    try:
        response = client.chat.completions.create(model=model, messages=messages, **kvargs)
        return response.choices[0].message.content
    except Exception as e:
        print(f'model {model} error: {e}')
# Ping!
messages = [user("Hello, how are you?")]
for model in ['glm', 'gpt-codex', 'gpt', 'gemini-flash', 'gemini-pro']:
    response = chat(model=model, messages=messages)
    print(f'model {model} response: {response}')
resp = client.responses.create(
    model="gpt-codex-max",
    input="Hello, how are you?",
)

print(f"responses api: {resp.output_text}")

model glm response: Hello! I'm doing well, thank you for asking. How can I help you today?
model gpt-codex response: Hi! I’m doing well—thanks for asking 😊  
How are you doing?
model gpt response: Hello! I’m doing well, thanks for asking. How can I help you today?
model gemini-flash response: I'm doing well, thank you for asking! How can I help you today?
model gemini-pro response: Hello! I'm doing well, thank you for asking. How are you doing today, and how can I help you?
responses api: Hello! I'm just a computer program, but I'm here and ready to help. What can I do for you today?


In [3]:
# 1. Find category id in miniflux by name
response = miniflux_client.get_categories()
economy_id = next((c['id'] for c in response if c['title'] == 'news'))
print(f'category id {economy_id}')
# 2. Get category entries
import datetime
now = datetime.datetime.now()
one_day_ago = now - datetime.timedelta(days=1)

start_timestamp = int(one_day_ago.timestamp())
from bs4 import BeautifulSoup

def pure_content(entry_content):
    soup = BeautifulSoup(entry_content, 'html.parser')
    return soup.get_text(" ", strip=True)
    
response = miniflux_client.get_category_entries(economy_id, 
                                            order='published_at',
                                            limit=10000,
                                            published_after=start_timestamp)
# Convert entries to Pandas
data = [{'id': e['id'],
         'Title': e['title'],
         'Link': e['url'],
         'Content': pure_content(e['content'])[:1000],
         'published_at': e['published_at'],
         'Source': e['feed']['title']} for e in response['entries']]
import pandas as pd
df = pd.DataFrame(data).set_index('id')
# print(df)
# 3. Format dataframe for AI user message
# content = df.to_xml(parser='etree')
from pathlib import Path
current_directory_path = Path().cwd()
print(f"Current working directory (Path object): {current_directory_path}")
# df.to_csv(current_directory_path / 'news-2025-09-29.csv', index=True)

#content = df.to_csv(index=False, columns=['Title', 'Content', 'Source', 'Link'])
#content = df.to_xml(row_name='news', pretty_print=False,
#                    xml_declaration=False, parser='etree',
#                    index=False, elem_cols=['Title', 'Content', 'Link'])
def format_row(index,row):
    return f"""# Entity {index}
Title: {row['Title']}
Content: {row['Content']}
Source: {row['Source']}
Link: {row['Link']}
"""
content = "\n".join((format_row(i, row) for i, row in df.iterrows()))
print(content[:100])
len(df)

category id 2
Current working directory (Path object): /home/asmirnov/work/ai-news
# Entity 566133
Title: Michael Avenatti, who stole from Stormy Daniels, released from prison
Content


1465

In [4]:
# Query perplexity.ai to generate examples about trending news
trending_news = chat(model='sonar-reasoning-pro',
                     messages=[user("What are the most trending US and world news for the last 24 hours")])
print(trending_news)

# Trending US and World News

## United States

**US-Iran ceasefire deal** tops headlines, with reports indicating the Trump administration negotiated a last-minute agreement[1]. Markets reacted positively, with the Dow closing up more than 1,300 points following the announcement[1]. The deal has sparked political debate, with Democrats calling for accountability over Trump's inflammatory rhetoric and some Republicans distancing themselves from his inflammatory statements[1].

**Artemis II splashdown** is scheduled for Friday with NASA providing updates Thursday on the re-entry process[2]. Weather conditions appear mild, though officials are monitoring for potential rain[2].

**Brian Hooker's wife disappearance case** in the Bahamas continues to develop, with Hooker arrested by the Royal Bahamas Police Force after his wife, Lynette, apparently fell overboard from their dinghy[2]. Exclusive Facebook messages between Hooker and a friend have been reviewed by CBS News[2].

Additional US s

In [5]:
instructions = f"""You are a news analyst.
  Your task is to create a comprehensive digest of events from provided news sources.
  Given media data format:
  Title: news headline
  Content: news summary
  Source: publisher
  Link: URL to original source
  Find the most trending news by identifying related events reported by multiple sources.
  Consider news related when they share persons, organizations, locations, or events.
  Use broad criteria for relations. For example, combine all legal actions by the president,
  or group economic news about the same technology, trend, or event.

  Pay special attention to:
  - President Trump's actions, lawsuits, and executive orders
  - Tariffs and their effects on the U.S. and world economy
  - Job market, especially related to AI technologies
  - War in Ukraine
  - Midterm elections and U.S. political parties
  - Bay Area news

  Combine each mention of the same news into a single record. Translate all texts to English.
  Provide all links for related news items.
Example of the possible trending news today:
{trending_news}
Output format:
<news>
<title>Summary title of the trending event</title>
<summary>Brief summary that describes the event:
What happend, why it matters, possible consequences
No more than 1 paragraph
</summary>
<link>https://example.com/article</link>
...
<link>https://foo.com/event.html</link>
</news>
....
"""
prompt = f"""This is the news data
<data>
{content}
</data>
Your answer:
<news>"""
messages = [{"content": instructions, "role": "system"},
            {"content": prompt, "role": "user"}]
text = chat(model='gemini-flash',
            messages=messages,
            reasoning_effort="high",
            temperature=1.0
            # extra_body={"generationConfig": {'thinkingBudget': -1}}
           )
print(text[:100])

<news>
<title>Fragile US-Iran Ceasefire Threatened by Israeli Strikes in Lebanon and Disputed Terms<


In [6]:
import json

import xml.etree.ElementTree as ET
import re
NEWS_BLOCK_RE = re.compile(r"<news\b[^>]*>.*?</news>", re.DOTALL | re.IGNORECASE)
def iter_news_blocks(text: str):
    """
    Yield each <news>...</news> block found in `text`.
    Uses a non-greedy regex so it works on fragments / multiple top-level elements.
    """
    for m in NEWS_BLOCK_RE.finditer(text):
        yield m.group(0)


def parse_news_block(block: str):
    """
    Parse a single <news>...</news> block and return a dict with title, summary, links.
    Tries xml.etree.ElementTree first; falls back to BeautifulSoup if available;
    otherwise uses regex heuristics.
    """
    # Try ElementTree
    try:
        root = ET.fromstring(block)
        title = root.findtext("title")
        summary = root.findtext("summary")
        links = [el.text for el in root.findall("link")]
        return {"title": title, "summary": summary, "links": links}
    except Exception:
        pass
# Last-resort regex extraction (very forgiving)
    def inner_text(tag: str):
        pattern = re.compile(fr"<{tag}[^>]*>(.*?)</{tag}>", re.DOTALL | re.IGNORECASE)
        return [m.group(1).strip() for m in pattern.finditer(block)]

    title_list = inner_text("title")
    summary_list = inner_text("summary")
    links = inner_text("link")
    return {
        "title": title_list[0] if title_list else None,
        "summary": summary_list[0] if summary_list else None,
        "links": links,
    }

records = [parse_news_block(b) for b in iter_news_blocks(text)]
print(len(records))

9


In [7]:
# refine information from source links
# use google grounded with web fetch
tools = [
  {"url_context": {}},
]
current_date = datetime.date.today()
def refine(title, content, links):
    model = 'gemini-flash'
    sys_prompt = f"""Generate summary about news record from provided sources.
Only consider information from the original sources, DO NOT invent any facts
Today date is {current_date.strftime("%Y-%m-%d")}, the current president of the United States is Donald Trump
Include sections:
# What hapenned
Provide more detailed summary here, including facts and opinions
# Why it matters
# What are possible consequences
# Contradictinal opinions ( if any )
"""
    msgs = [system(sys_prompt),
            user(f"# {title}\n\n{content}\nsources: {' '.join(links[:20])}")]
    summary = chat(model=model,
                   messages=msgs, tools=tools, extra_body={'thinkingBudget': -1})
    return summary or "EMPTY"
# print(refine(records[0]['title'], records[0]['links']))

In [8]:
from IPython.display import display, HTML
import json, html
style = """<style>
    .jp-CodeCell.jp-mod-outputsScrolled .jp-Cell-outputArea {
        max-height: unset !important; /* Remove the max-height limit */
        overflow-y: visible !important; /* Allow content to overflow vertically */
    }
    div.output_scroll {
        height: fit-content !important; /* Make the output scroll container fit its content */
        overflow-y: visible !important; /* Allow content to overflow vertically */
    }
</style>"""
def display_records_collapsible(records):
    parts = []
    for i, r in enumerate(records):
        title = html.escape(r.get("title") or f"record {i}")
        # summary = html.escape(r.get('summary'))
        summary = html.escape(refine(r.get('title'), r.get('summary'), r.get('links')))
        links = [f'<li><a href="{link}">{link}</a></li>' for link in r.get('links')]
        parts.append(f"""
        <details style="margin:8px 0; padding:6px; border:1px solid #ddd; border-radius:4px;">
          <summary style="font-weight:600; cursor:pointer;">{title}</summary>
          <p style="white-space:pre-wrap; margin:8px 0;">{summary}</p>
          <ul>{''.join(links)}</ul>
        </details>
        """)
    html_doc = style + "<div>" + "\n".join(parts) + "</div>"
    display(HTML(html_doc))

display_records_collapsible(records)